# Chapter 14: Beyond Simulation
Run these six cells in order in the Appendix IV Qiskit environment.
Default execution is entirely local; physical hardware is optional and was not run.
Records are written to `chapter14_results/` relative to the notebook working directory.
Use a fresh directory for each run. The manuscript explains the assumptions and exit criteria.


In [ ]:
import json
import time
import platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, transpile, qasm3
from qiskit.primitives import BackendSamplerV2
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error

SHOTS, REPEATS, SEED = 2048, 8, 14
OUT = Path("chapter14_results")
OUT.mkdir(exist_ok=True)
circuit = QuantumCircuit(2)
circuit.ry(np.pi/3, 0)
circuit.ry(np.pi/4, 1)
circuit.cx(0, 1)
exact = float(Statevector.from_instruction(circuit).expectation_value(
    SparsePauliOp("ZI")).real)  # Qiskit: q0 is the rightmost bit.
assert np.isclose(exact, np.cos(np.pi/3)*np.cos(np.pi/4))
measured = circuit.copy()
measured.measure_all()
control = QuantumCircuit(2)
control.measure_all()
noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.12, 2), ["cx"])
noise.add_all_qubit_readout_error(ReadoutError([[.92, .08], [.08, .92]]))
ideal_backend = AerSimulator()
noisy_backend = AerSimulator(noise_model=noise)
print("Versions:", qiskit.__version__, qiskit_aer.__version__)
print("Exact Z1:", round(exact, 6))


In [ ]:
def utc_now():
    """Return an ISO timestamp for the execution ledger."""
    return datetime.now(timezone.utc).isoformat()


def execute_record(backend, sampler, circuits, name, shots, layout=None, seed=None):
    """Compile and submit once, persisting metadata and any failure."""
    path = OUT / (name + ".json")
    record = {"name": name, "backend": backend.name,
              "created_utc": utc_now(), "status": "prepared",
              "requested_shots_per_circuit": shots,
              "completed_shots": 0, "retries": 0, "failures": [],
              "job_id": None, "provider_metrics": None,
              "seed_simulator": seed,
              "optimization_level": 0, "seed_transpiler": SEED}
    def save():
        path.write_text(json.dumps(record, indent=2, default=str))
    start = time.perf_counter()
    try:
        compiled = transpile(circuits, backend, optimization_level=0,
                             seed_transpiler=SEED, initial_layout=layout)
        record["circuits"] = [{
            "depth": c.depth(), "operations": dict(c.count_ops()),
            "two_qubit_gates": sum(len(i.qubits) == 2 for i in c.data),
            "physical_qubits": (c.layout.final_index_layout() if c.layout
                                else list(range(c.num_qubits))),
            "qasm": qasm3.dumps(c),
        } for c in compiled]
        if any(c["depth"] > 20 or c["two_qubit_gates"] > 4
               for c in record["circuits"]):
            raise ValueError("Compilation exceeds the pilot cost limit")
        save()
        job = sampler.run(compiled, shots=shots)
        record.update(job_id=job.job_id(), submitted_utc=utc_now())
        save()  # Keep the ID even if result retrieval later fails.
        result = job.result()
        counts = [pub.data.meas.get_counts() for pub in result]
        record["counts"] = counts
        record["completed_shots"] = sum(sum(c.values()) for c in counts)
        record["result_metadata"] = [pub.metadata for pub in result]
        if len(counts) != len(circuits) or any(
                sum(c.values()) != shots for c in counts):
            raise ValueError("Incomplete result; do not silently analyze it")
        if hasattr(job, "metrics"):
            record["provider_metrics"] = job.metrics()
        record["status"] = "completed"
    except Exception as error:
        record["status"] = "failed"
        record["failures"].append(type(error).__name__)
    record.update(finished_utc=utc_now(),
                  wall_seconds=time.perf_counter()-start)
    save()
    return record


In [ ]:
def joint_rows(counts):
    """Expand joint counts into paired (Z0, Z1) outcomes, without time order."""
    rows = []
    for bits, count in counts.items():
        z = [1-2*int(bits[-1]), 1-2*int(bits[-2])]
        rows.extend([z]*count)
    return np.asarray(rows, dtype=float)


records, samples = {}, {}
for stage, backend, reference in (
        ("finite", ideal_backend, exact),
        ("noisy", noisy_backend, exact*.88*.84)):
    records[stage], samples[stage] = [], []
    for repeat in range(REPEATS):
        sampler = BackendSamplerV2(backend=backend, options={
            "seed_simulator": SEED + repeat})
        rec = execute_record(backend, sampler, [measured, control],
                             f"{stage}_{repeat}", SHOTS, [0, 1], SEED+repeat)
        records[stage].append(rec)
        if rec["status"] != "completed":
            raise RuntimeError("Stop and inspect the saved failed record")
        samples[stage].append(joint_rows(rec["counts"][0]))
    pooled = np.concatenate(samples[stage])
    mean = pooled[:, 1].mean()
    se = pooled[:, 1].std(ddof=1)/np.sqrt(len(pooled))
    assert abs(mean-reference) < 5*se
    print(stage, "Z1/SE:", round(mean, 6), round(se, 6))
    controls = np.concatenate([joint_rows(r["counts"][1])
                               for r in records[stage]])[:, 1]
    control_target = 1. if stage == "finite" else .84
    control_se = controls.std(ddof=1)/np.sqrt(len(controls))
    assert abs(controls.mean()-control_target) <= 5*control_se + 1e-12
    print(stage, "control Z1:", round(controls.mean(), 6))
print("Noisy analytic Z1:", round(exact*.88*.84, 6))


In [ ]:
rows = np.concatenate(samples["noisy"])
weights = np.array([.5, .5])
covariance = np.cov(rows, rowvar=False, ddof=1)
group_se = np.sqrt(weights @ covariance @ weights / len(rows))
independent_se = np.sqrt(np.sum(weights**2*np.diag(covariance))/len(rows))
print("Grouped SE / ignoring covariance:",
      round(group_se, 6), round(independent_se, 6))

# For independent, representative time windows, resample entire jobs.
job_means = np.array([r[:, 1].mean() for r in samples["noisy"]])
bootstrap_rng = np.random.default_rng(140)
job_bootstrap = bootstrap_rng.choice(job_means, (2000, REPEATS)).mean(1)
print("Job-bootstrap interval:",
      np.round(np.quantile(job_bootstrap, [.025, .975]), 6))
variances = np.array([.81, .09])  # Separate illustrative pilot groups.
allocation = 400*np.sqrt(variances)/np.sqrt(variances).sum()
print("Illustrative group allocations:", allocation.astype(int))


In [ ]:
calibration = []
for value in range(4):
    c = QuantumCircuit(2)
    for wire in range(2):
        if (value >> wire) & 1:
            c.x(wire)
    c.measure_all()
    calibration.append(c)
sampler = BackendSamplerV2(backend=noisy_backend,
                           options={"seed_simulator": 141})
cal = execute_record(noisy_backend, sampler, calibration,
                     "readout_calibration", 4096, [0, 1], 141)
if cal["status"] != "completed":
    raise RuntimeError("Calibration failed; inspect the ledger")


def frequencies(counts):
    """Return frequencies in Qiskit order 00, 01, 10, 11."""
    return np.array([counts.get(f"{i:02b}", 0) for i in range(4)]) / sum(counts.values())


A = np.column_stack([frequencies(c) for c in cal["counts"]])
assert np.linalg.cond(A) < 10
raw_counts = {f"{i:02b}": sum(r["counts"][0].get(f"{i:02b}", 0)
              for r in records["noisy"]) for i in range(4)}
raw = frequencies(raw_counts)
z1 = np.array([1., 1., -1., -1.])
quasi = np.linalg.solve(A, raw)
mitigated = float(z1 @ quasi)
rng = np.random.default_rng(142)
replicates = []
for _ in range(2000):
    A_star = np.column_stack([
        rng.multinomial(4096, A[:, j])/4096 for j in range(4)])
    raw_star = rng.multinomial(sum(raw_counts.values()), raw)/sum(raw_counts.values())
    if np.linalg.cond(A_star) >= 10:
        raise RuntimeError("Unstable calibration bootstrap; revise the protocol")
    replicates.append(z1 @ np.linalg.solve(A_star, raw_star))
interval = np.quantile(replicates, [.025, .975])
print("Readout-mitigated Z1/interval:", round(mitigated, 6),
      np.round(interval, 6))
print("Gate-noisy target:", round(exact*.88, 6))


In [ ]:
hardware_record = {
    "status": "not_run", "reason": "No physical backend configured",
    "backend": None, "calibration": None, "job_id": None,
    "requested_shots": 0, "completed_shots": 0,
    "failures": [], "retries": 0, "queue_seconds": None,
}


def optional_hardware(backend, sampler, physical_qubits):
    """Submit one bounded pilot using an already configured provider sampler."""
    if (OUT / "hardware_pilot.json").exists():
        raise RuntimeError("Inspect the existing pilot before resubmitting")
    if len(physical_qubits) != 2 or len(set(physical_qubits)) != 2:
        raise ValueError("Choose two distinct physical qubits")
    context = {"backend": backend.name, "captured_utc": utc_now(),
               "num_qubits": backend.num_qubits,
               "basis_gates": list(backend.operation_names),
               "coupling_map": str(backend.coupling_map),
               "selected_qubits": list(physical_qubits),
               "calibration": None}
    if hasattr(backend, "properties"):
        properties = backend.properties()
        context["calibration"] = properties.to_dict() if properties else None
    (OUT / "hardware_context.json").write_text(
        json.dumps(context, indent=2, default=str))
    return execute_record(backend, sampler, [measured, control],
                          "hardware_pilot", 1024, physical_qubits)


# Optional: hardware_record = optional_hardware(backend, sampler, [q0, q1])
artifact = {
    "versions": {"python": platform.python_version(),
                 "numpy": np.__version__, "qiskit": qiskit.__version__,
                 "aer": qiskit_aer.__version__},
    "seed": SEED, "shots_per_circuit": SHOTS, "repeats": REPEATS,
    "noise": {"cx_depolarizing": .12, "readout_flip": .08},
    "exact_Z1": exact, "records": records, "calibration": cal,
    "mitigated_Z1": mitigated, "mitigation_interval": interval.tolist(),
    "hardware": hardware_record,
}
(OUT / "summary.json").write_text(json.dumps(artifact, indent=2, default=str))
total = sum(r["completed_shots"] for v in records.values() for r in v)
print("Completed local shots:", total + cal["completed_shots"])
print("Hardware status:", hardware_record["status"])
